In [ ]:
pip install requests

In [1]:
"""
Sube una estructura de carpetas local completa a un repositorio de GitHub,
manteniendo la jerarquía de carpetas y subcarpetas.

Requisitos:
    pip install requests

Uso:
    python subir_a_github.py

Antes de ejecutar, completá las variables de configuración de más abajo.
"""

import os
import base64
import requests
from google.colab import drive
import os

drive.mount('/content/drive')

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    import getpass
    GITHUB_TOKEN = getpass.getpass("Pegá tu GitHub Personal Access Token: ")



print("✅ README.md ampliado generado correctamente.")



# ============ CONFIGURACIÓN ============
OWNER = "robertodavidalcoba-design"       # Usuario u organización dueño del repo
REPO = "Curso-DataAnalitic-CoderHouse"          # Nombre del repositorio
BRANCH = "main"                           # Rama destino
LOCAL_FOLDER = "/content/drive/MyDrive/Trabajo Final Data2"    # Carpeta local que querés subir
REMOTE_BASE_PATH = ""                     # Carpeta destino dentro del repo ("" = raíz)

%cd {LOCAL_FOLDER}
contenido="""# Predicción de Rendimiento Académico (Trabajo Final — Data Science II)

Pipeline de Machine Learning para predecir la **nota final** (`nota_final`, escala 0–20) de estudiantes de dos escuelas secundarias de Portugal (Gabriel Pereira y Mousinho da Silveira), a partir de variables socioeconómicas, hábitos y entorno familiar — **sin usar las notas parciales previas** (G1/G2) como predictoras, para evitar fuga de información y evaluar el valor predictivo real de los factores de contexto.

**Autor:** Roberto David Alcoba

---

## 1. Problema y objetivo

- **Tarea:** Regresión.
- **Variable objetivo:** `nota_final` (G3 original), rango 0–20.
- **Métricas de éxito:** R², R² ajustado, RMSE y MAE sobre un conjunto *holdout* (20%) no visto durante el entrenamiento.
- **Relevancia de negocio:** el modelo busca servir como señal complementaria de alerta temprana para priorizar intervención pedagógica (tutorías, apoyo socioeconómico), no como reemplazo del criterio docente. Por eso se excluyen deliberadamente `nota_periodo1` y `nota_periodo2` (G1/G2): incluirlas resuelve el problema de forma casi trivial (están altamente correlacionadas con G3) pero no aporta valor de negocio, ya que para cuando existen esas notas parciales gran parte del período académico ya transcurrió.

## 2. Dataset

- **Fuente:** [Student Performance Dataset (UCI Machine Learning Repository)](https://archive.ics.uci.edu/dataset/320/student+performance), recolectado en escuelas secundarias de Portugal.
- **Archivo esperado:** `data/student_data.csv`
- **Filas / columnas originales:** 395 estudiantes, 33 variables (renombradas al español en el pipeline).
- **Escuelas:** `GP` = Gabriel Pereira (Évora), `MS` = Mousinho da Silveira (Portalegre).

> El dataset no se distribuye en este repositorio por tamaño/licencia. Descargarlo del enlace de UCI y colocarlo en `data/student_data.csv` antes de ejecutar el pipeline (ver sección 5).

## 3. Estructura del repositorio
```text
project-root/
│
├── README.md           # Documentación del proyecto
├── requirements.txt    # Librerías necesarias
├── outputs/          # Notebooks ordenados por etapa
├── scripts/            # Módulos Python reutilizables (utils.py)
├── data/               # Datasets de entrada
└── doc/                # reportes Informe Final, Resultados del Pipeline
├── data/               # Datasets de entrada
└── img/                # Gráficos y reportes generados

## 4. Requisitos

- Python 3.10+
- Dependencias principales:

```
pandas
numpy
matplotlib
seaborn
scipy
statsmodels
scikit-learn
xgboost
lightgbm
shap
joblib
```

Instalación:

```bash
pip install -r requirements.txt
```

> `xgboost` y `lightgbm` son opcionales: si no están instalados, el pipeline continúa y salta esos modelos con un aviso por consola.

## 5. Cómo ejecutar el proyecto

### Opción A — Notebook (Google Colab)

El notebook fue desarrollado originalmente en Google Colab y monta Google Drive para localizar automáticamente la raíz del proyecto (carpetas `data/`, `doc/`, `img/`, `outputs/`, `scripts/`) a partir del propio nombre del `.ipynb`.

1. Subir la carpeta completa del proyecto (con `data/student_data.csv` incluido) a Google Drive.
2. Abrir `Trabajo Final DataII_RobertoDavidAlcoba.ipynb` en Colab.
3. Ejecutar todas las celdas (`Entorno de ejecución → Ejecutar todas`). La primera celda solicitará autorización para montar Drive.

### Opción B — Local / fuera de Colab

El script depende de `google.colab.drive`, por lo que para correrlo localmente hay que reemplazar el bloque de montaje de Drive por una ruta local fija. Pasos sugeridos:

1. Clonar el repositorio y colocar el dataset en `data/student_data.csv`.
2. En `scripts/pipeline_trabajo_final.py`, sustituir:
   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   ```
   por la definición directa del diccionario `paths` apuntando a las carpetas locales del repo (`data/`, `doc/`, `img/`, `outputs/`, `scripts/`).
3. Ejecutar:
   ```bash
   python scripts/pipeline_trabajo_final.py
   ```

Al finalizar, el pipeline genera en `outputs/`:
- CSV con el ranking de variables (`f_regression` vs `mutual_info_regression`).
- Modelo entrenado serializado: `modelo_lineal_nota_final.pkl` (pipeline completo: preprocesamiento + `LinearRegression`).
- Gráficos de diagnóstico y SHAP en `img/` e `img/shap/`.

## 6. Resumen del pipeline

1. **Carga y estandarización:** lectura de `student_data.csv`, renombre de 33 columnas al español.
2. **EDA y detección de atípicos:** análisis univariado del target, IQR y puntaje Z, revisión de consistencia de categorías.
3. **Codificación categórica:** binarias/ordinales por mapeo numérico, nominales por One-Hot Encoding (`drop_first=True`).
4. **Escalado híbrido:** `RobustScaler` para variables con `|skew| > 1.0` (ej. ausencias), `MinMaxScaler` para el resto.
5. **Feature engineering:** creación de 4 índices sintéticos (`indice_riesgo_academico`, `nivel_educativo_parental`, `indice_riesgo_social`, `indice_compromiso_academico`).
6. **Análisis bivariado y VIF:** verificación de ausencia de multicolinealidad severa (VIF < 2.0).
7. **Selección de variables:** dos *feature sets* — Set A (10 variables, `f_regression`, familia lineal) y Set B (13 variables, `f_regression` + `mutual_info_regression`, familia de ensambles).
8. **Entrenamiento y optimización:**
   - Familia lineal: `LinearRegression`, `RidgeCV`, `LassoCV` sobre Set A.
   - Familia de ensambles: `RandomForestRegressor`, `XGBRegressor`, `LGBMRegressor` sobre Set B, tuneados con `RandomizedSearchCV` (`n_iter=25`) — se prefirió sobre `GridSearchCV` por el tamaño combinado del espacio de hiperparámetros de los tres modelos, con costo computacional acotado.
   - Validación: K-Fold (`cv=5`) + *holdout* 80/20.
9. **Diagnóstico de residuos:** pruebas de Shapiro-Wilk (normalidad), Breusch-Pagan (homocedasticidad) y Durbin-Watson (independencia) sobre el modelo lineal ganador.
10. **Explicabilidad (SHAP):** `shap.LinearExplainer` y `shap.TreeExplainer`, *summary plots* guardados por modelo.
11. **Exportación:** pipeline completo (preprocesador + modelo) serializado con `joblib`.

## 7. Resultados (holdout 20%)

| Familia | Feature Set | Modelo | R² | R² ajustado | RMSE | MAE |
|---|---|---|---|---|---|---|
| Lineal | A | **Linear Regression** | **0.112** | -0.018 | 0.213 | 0.167 |
| Lineal | A | Lasso (LassoCV) | 0.106 | -0.026 | 0.214 | 0.168 |
| Ensamble | B | Random Forest | 0.098 | -0.083 | 0.215 | 0.172 |
| Lineal | A | Ridge (RidgeCV) | 0.089 | -0.045 | 0.216 | 0.170 |
| Ensamble | B | XGBoost | 0.068 | -0.119 | 0.219 | 0.176 |
| Ensamble | B | LightGBM | -0.081 | -0.298 | 0.235 | 0.198 |

*RMSE y MAE están en la escala procesada/escalada.*

**Modelo elegido:** `LinearRegression` sobre Feature Set A. Nota importante: al excluir G1/G2 del conjunto de predictores, el poder explicativo es intencionalmente bajo (R² ajustado negativo en todos los modelos); el valor del proyecto está en la interpretabilidad de los coeficientes/SHAP más que en la precisión predictiva pura. Se recomienda usarlo como señal complementaria de priorización, no como decisión automática.

## 8. Explicabilidad — hallazgos principales (SHAP)

- `indice_riesgo_academico` (materias reprobadas, ausencias severas, tiempo de estudio) es el factor con mayor impacto **negativo** sobre la nota final.
- `nivel_educativo_parental` y `clases_particulares_pagadas` son los factores protectores con mayor impacto **positivo**.
- `indice_riesgo_social` (alcohol, salidas, relación amorosa) tiene impacto negativo moderado pero constante.

Ver gráficos en `img/shap/`.

## 9. Licencia y créditos

Dataset: P. Cortez y A. Silva, *Using Data Mining to Predict Secondary School Student Performance*, UCI Machine Learning Repository.
Uso académico — Trabajo Final, materia Data Science II.


"""

with open("README.md", "w") as f:
    f.write(contenido)
f.close()


# Carpetas/archivos a ignorar
IGNORAR = {".git", "__pycache__", ".DS_Store", "node_modules", ".venv"}
# ========================================

API_URL = "https://api.github.com"

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
}


def obtener_sha_existente(ruta_remota):
    """Si el archivo ya existe en el repo, devuelve su sha (necesario para actualizarlo)."""
    url = f"{API_URL}/repos/{OWNER}/{REPO}/contents/{ruta_remota}"
    resp = requests.get(url, headers=HEADERS, params={"ref": BRANCH})
    if resp.status_code == 200:
        return resp.json().get("sha")
    return None


def subir_archivo(ruta_local, ruta_remota):
    """Sube (o actualiza) un archivo individual al repositorio."""
    with open(ruta_local, "rb") as f:
        contenido = f.read()

    contenido_b64 = base64.b64encode(contenido).decode("utf-8")
    sha = obtener_sha_existente(ruta_remota)

    data = {
        "message": f"Subir {ruta_remota}",
        "content": contenido_b64,
        "branch": BRANCH,
    }
    if sha:
        data["sha"] = sha  # requerido si el archivo ya existe

    url = f"{API_URL}/repos/{OWNER}/{REPO}/contents/{ruta_remota}"
    resp = requests.put(url, headers=HEADERS, json=data)

    if resp.status_code in (200, 201):
        print(f"✅ Subido: {ruta_remota}")
    else:
        print(f"❌ Error en {ruta_remota}: {resp.status_code} - {resp.json().get('message')}")


def recorrer_y_subir(carpeta_local, base_remota):
    """Recorre recursivamente la carpeta local y sube cada archivo respetando la jerarquía."""
    for raiz, carpetas, archivos in os.walk(carpeta_local):
        carpetas[:] = [c for c in carpetas if c not in IGNORAR]

        for archivo in archivos:
            if archivo in IGNORAR:
                continue

            ruta_local = os.path.join(raiz, archivo)
            ruta_relativa = os.path.relpath(ruta_local, carpeta_local)
            ruta_relativa = ruta_relativa.replace(os.sep, "/")  # rutas estilo GitHub

            ruta_remota = f"{base_remota}/{ruta_relativa}" if base_remota else ruta_relativa

            subir_archivo(ruta_local, ruta_remota)


if __name__ == "__main__":
    if not GITHUB_TOKEN:
        print("⚠️  No se encontró el token. Configurá el secreto GITHUB_TOKEN en Colab (🔑) o pegalo cuando se te pida.")
    elif not os.path.exists(LOCAL_FOLDER):
        print(f"❌ La carpeta '{LOCAL_FOLDER}' no existe. Verificá la ruta (ruta absoluta recomendada).")
    else:
        # Contar archivos antes de subir, para detectar carpetas vacías o mal apuntadas
        total_archivos = sum(
            1 for _, _, archivos in os.walk(LOCAL_FOLDER) for a in archivos if a not in IGNORAR
        )
        if total_archivos == 0:
            print(f"⚠️  No se encontró ningún archivo dentro de '{LOCAL_FOLDER}'. Nada para subir.")
        else:
            print(f"📁 Se encontraron {total_archivos} archivo(s) en '{LOCAL_FOLDER}'. Subiendo...")
            recorrer_y_subir(LOCAL_FOLDER, REMOTE_BASE_PATH)
            print("🎉 Proceso terminado.")

Mounted at /content/drive
Pegá tu GitHub Personal Access Token: ··········
✅ README.md ampliado generado correctamente.
/content/drive/MyDrive/Trabajo Final Data2
📁 Se encontraron 132 archivo(s) en '/content/drive/MyDrive/Trabajo Final Data2'. Subiendo...
✅ Subido: Trabajo Final DataII_RobertoDavidAlcoba.ipynb
✅ Subido: README.md
✅ Subido: requirements.txt
✅ Subido: Trabajo Final DataII a Github.ipynb
✅ Subido: data/student_data.csv
✅ Subido: doc/Resultado_Pipeline_Trabajo_ Final_DataII.docx
✅ Subido: doc/Seleccion_de_Variables_Candidatas.xlsx
✅ Subido: doc/Formulacion_del_problema.docx
✅ Subido: doc/Informe_Final_Trabajo_Data_II.docx
✅ Subido: doc/Informe Final — Trabajo Final Data Science II (Versión Actualizada).docx
✅ Subido: img/analisis_univariado_target.png
✅ Subido: img/boxplots_outliers.png
✅ Subido: img/heatmap_correlacion.png
✅ Subido: img/boxplots_categoricas_vs_target.png
✅ Subido: img/scatter_vs_target.png
✅ Subido: img/residuos_Linear_Regression.png
✅ Subido: img/residu